# Polypyrrole Actuator Analysis — 2026-03-19 Actuation 2

Combines:
- **CSV**: `2026.3.19-actuation2.csv` — current (µA) vs time (s) from potentiostat
- **Video**: `2026.3.19-actuation2_voltage_out.mp4` — voltage readout from analog meter (64×24 px, ~23 fps)

Pipeline:
1. Extract voltage from video via OCR (green-channel isolation → threshold → `letsgodigital` Tesseract model)
2. Load both sources, apply a **time offset** to synchronize, merge, and plot

In [1]:
import cv2
import numpy as np
import pandas as pd
import pytesseract
import re
from pathlib import Path
from tqdm.auto import tqdm

pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'

VIDEO_PATH   = Path('2026.3.19-actuation2_voltage_out.mp4')
CSV_PATH     = Path('2026.3.19-actuation2.csv')
VOLTAGE_CSV  = Path('2026.3.19-actuation2_voltage_ocr.csv')  # output

# Decimal position: number of digits from the right before the decimal point
# e.g. DECIMAL_FROM_RIGHT=3 → 10998 → 10.998
DECIMAL_FROM_RIGHT = 3

print('OpenCV version:', cv2.__version__)
print('Tesseract:', pytesseract.get_tesseract_version())

OpenCV version: 4.13.0
Tesseract: 5.5.1


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Extract voltage readings from video

Samples one frame per 0.1 s. For each timestamp, averages up to 3 consecutive raw frames before OCR to reduce noise.  
Bad reads (non-numeric, out-of-range) are set to `NaN` and later interpolated.

---
## Alternative: digit-crop → manual label → PCA nearest-neighbour classifier

Better than raw Tesseract for 7-segment displays at low resolution.

**Workflow:**
1. Run *Step A* → extracts every digit from every frame into `digit_crops/all_frames/`  
   and a smaller sample into `digit_crops/to_label/` (one per 3 s)
2. **You manually** drag crops from `to_label/` into `digit_crops/labeled/0/` … `9/`  
   (only need ~5–10 good examples per digit)
3. Run *Step B* → PCA + nearest-neighbour classifies all frames → new clean voltage CSV

In [ ]:
### Step A — extract digit crops from every frame + labeling sample

from pathlib import Path
import cv2, numpy as np

# ── Column boundaries for each digit (x_start, x_end) in the 64-px-wide frame ──
# D0=tens, D1=units, D2=first decimal, D3=second decimal, D4=third decimal
DIGIT_COLS = [
    (5,  14),   # D0  (e.g. "1" in  1x.xxx)
    (18, 27),   # D1  (e.g. "0" in  10.xxx)
    (27, 40),   # D2  first decimal digit
    (39, 52),   # D3  second decimal digit
    (51, 64),   # D4  third decimal digit
]
CROP_W, CROP_H  = 24, 48   # upscaled output size (nearest-neighbour)
LABEL_EVERY_S   = 3.0      # one sample per N seconds saved to to_label/
ALL_EVERY_S     = 0.1      # one sample per N seconds saved to all_frames/ (= every OCR timestamp)

crops_root  = Path('digit_crops')
label_dir   = crops_root / 'to_label'
all_dir     = crops_root / 'all_frames'
labeled_dir = crops_root / 'labeled'

label_dir.mkdir(parents=True, exist_ok=True)
all_dir.mkdir(parents=True, exist_ok=True)
for d in range(10):
    (labeled_dir / str(d)).mkdir(parents=True, exist_ok=True)


def extract_digit_crop(frame_gray, x0, x1, w=CROP_W, h=CROP_H):
    """Threshold green channel, crop digit column, upscale."""
    _, thresh = cv2.threshold(frame_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    crop = thresh[:, x0:x1]
    return cv2.resize(crop, (w, h), interpolation=cv2.INTER_NEAREST)


cap = cv2.VideoCapture(str(VIDEO_PATH))
fps  = cap.get(cv2.CAP_PROP_FPS)
dur  = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
print(f"Video: {dur:.0f} s  |  {fps:.1f} fps")

n_label, n_all = 0, 0
t = 0.0
next_label = 0.0

while t <= dur:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ret, frame = cap.read()
    if not ret:
        t += ALL_EVERY_S
        continue

    g = frame[:, :, 1]
    fid = f"{t:08.3f}"

    for d, (x0, x1) in enumerate(DIGIT_COLS):
        crop = extract_digit_crop(g, x0, x1)
        cv2.imwrite(str(all_dir / f"t{fid}_d{d}.png"), crop)
        n_all += 1

        if t >= next_label:
            cv2.imwrite(str(label_dir / f"t{fid}_d{d}.png"), crop)
            n_label += 1

    if t >= next_label:
        next_label += LABEL_EVERY_S
    t = round(t + ALL_EVERY_S, 3)

cap.release()
print(f"Saved {n_all} crops to all_frames/  ({n_all // len(DIGIT_COLS)} timestamps)")
print(f"Saved {n_label} crops to to_label/  (label these, then run Step B)")
print(f"\nNow: drag crops from digit_crops/to_label/ into digit_crops/labeled/0/ … 9/")

In [ ]:
### Step B — train PCA + nearest-neighbour classifier on your labeled crops,
###           classify all frames, reconstruct voltage CSV

from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

N_COMPONENTS = 20   # PCA dims — raise if accuracy is low, lower if overfitting
N_NEIGHBORS  = 3    # KNN neighbours

# ── 1. Load labeled examples ─────────────────────────────────────────────────
X_train, y_train = [], []
for digit_dir in sorted(labeled_dir.iterdir()):
    if not digit_dir.is_dir() or not digit_dir.name.isdigit():
        continue
    label = int(digit_dir.name)
    for img_path in digit_dir.glob('*.png'):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        X_train.append(img.flatten().astype(np.float32) / 255.0)
        y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)
print(f"Labeled examples: {len(X_train)} crops")
for d in sorted(set(y_train)):
    print(f"  digit {d}: {(y_train==d).sum()} examples")

# ── 2. Fit PCA + KNN ─────────────────────────────────────────────────────────
pca = PCA(n_components=min(N_COMPONENTS, len(X_train) - 1), whiten=True)
X_pca = pca.fit_transform(X_train)
knn = KNeighborsClassifier(n_neighbors=N_NEIGHBORS, metric='euclidean')
knn.fit(X_pca, y_train)

print(f"\nPCA: {pca.n_components_} components, "
      f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance explained")

# Plot PCA scatter of training data
fig, ax = plt.subplots(figsize=(8, 6))
for d in sorted(set(y_train)):
    mask = y_train == d
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=str(d), s=40, alpha=0.7)
ax.set(xlabel='PC1', ylabel='PC2', title='PCA of labeled digit crops')
ax.legend(title='digit', ncol=2)
plt.tight_layout(); plt.show()

# ── 3. Classify all frames ────────────────────────────────────────────────────
all_crops = sorted(all_dir.glob('*.png'))
# Parse filename: t{time}_d{digit_pos}.png
records = {}   # key: (time_s, digit_pos) → predicted digit

print(f"\nClassifying {len(all_crops)} crops...")
batch_size = 500
imgs, keys = [], []
for path in all_crops:
    stem = path.stem   # e.g. t0014.500_d2
    parts = stem.split('_d')
    t_s = float(parts[0][1:])
    d_pos = int(parts[1])
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    imgs.append(img.flatten().astype(np.float32) / 255.0)
    keys.append((t_s, d_pos))

X_all = np.array(imgs)
X_all_pca = pca.transform(X_all)
preds = knn.predict(X_all_pca)

for (t_s, d_pos), pred in zip(keys, preds):
    if t_s not in records:
        records[t_s] = {}
    records[t_s][d_pos] = pred

# ── 4. Reconstruct voltage from digits ───────────────────────────────────────
DIGIT_WEIGHTS = [10.0, 1.0, 0.1, 0.01, 0.001]  # D0×10 + D1×1 + D2×0.1 + ...

rows = []
for t_s in sorted(records):
    digits = records[t_s]
    if len(digits) < len(DIGIT_COLS):
        rows.append({'time_s': t_s, 'voltage_V': np.nan})
        continue
    voltage = sum(digits[i] * DIGIT_WEIGHTS[i] for i in range(len(DIGIT_COLS)))
    rows.append({'time_s': t_s, 'voltage_V': round(voltage, 4)})

df_v2 = pd.DataFrame(rows)

# Quick sanity: histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].hist(df_v2['voltage_V'].dropna(), bins=50, color='seagreen')
axes[0].set(xlabel='Voltage (V)', title='Raw PCA-classified voltage distribution')
axes[1].plot(df_v2['time_s'], df_v2['voltage_V'], lw=0.6, color='seagreen')
axes[1].set(xlabel='Time (s)', ylabel='Voltage (V)', title='Voltage vs time (raw, no offset)')
plt.tight_layout(); plt.show()

df_v2.head(10)

In [ ]:
### Step C — save PCA-classified voltage CSV (use this instead of the Tesseract one)
# Overrides VOLTAGE_CSV so the sync/merge cells below pick it up automatically

df_voltage = df_v2.copy()
df_voltage.to_csv(VOLTAGE_CSV, index=False)
print(f"Saved PCA voltage → {VOLTAGE_CSV}")
print(df_voltage['voltage_V'].describe().to_string())

In [ ]:
TESSERACT_CFG = '--oem 1 --psm 7 -l letsgodigital -c tessedit_char_whitelist=0123456789.-+'
VOLTAGE_MIN, VOLTAGE_MAX = -30.0, 30.0   # plausible range — adjust if needed
FRAME_AVERAGE_N = 3   # how many nearby frames to average before OCR


def preprocess_frame(frame_gray: np.ndarray, scale: int = 12) -> np.ndarray:
    """Isolate green channel, threshold at native res, upscale with NEAREST, small dilation."""
    _, thresh = cv2.threshold(frame_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    up = cv2.resize(thresh, (thresh.shape[1] * scale, thresh.shape[0] * scale),
                    interpolation=cv2.INTER_NEAREST)
    kernel = np.ones((3, 3), np.uint8)
    return cv2.dilate(up, kernel, iterations=1)


def ocr_frame(proc: np.ndarray) -> float | None:
    """Run Tesseract on preprocessed frame, return float or None."""
    raw = pytesseract.image_to_string(proc, config=TESSERACT_CFG).strip().rstrip('.')

    # Accept only a leading optional minus, then digits, optional single '.', then digits
    m = re.match(r'^(-?)([0-9]+)(?:\.([0-9]*))?$', raw)
    if not m:
        return None

    sign, int_part, frac_part = m.group(1), m.group(2), m.group(3) or ''
    all_digits = int_part + frac_part

    # Must have 2–6 total digits
    if not (2 <= len(all_digits) <= 6):
        return None

    if frac_part:
        # Decimal already placed by OCR — use directly
        try:
            val = float(f'{sign}{int_part}.{frac_part}')
        except ValueError:
            return None
    else:
        # No decimal in raw output — insert DECIMAL_FROM_RIGHT digits from the right
        d = len(all_digits)
        if d > DECIMAL_FROM_RIGHT:
            val_str = f'{sign}{all_digits[:-DECIMAL_FROM_RIGHT]}.{all_digits[-DECIMAL_FROM_RIGHT:]}'
        else:
            val_str = f'{sign}0.{all_digits.zfill(DECIMAL_FROM_RIGHT)}'
        try:
            val = float(val_str)
        except ValueError:
            return None

    return val if VOLTAGE_MIN <= val <= VOLTAGE_MAX else None


def extract_voltage_from_video(video_path: Path, dt: float = 0.1) -> pd.DataFrame:
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    print(f'Video: {duration:.1f} s  |  {fps:.2f} fps  |  {total_frames} frames')

    timestamps = np.arange(0, duration, dt)
    records = []

    for t in tqdm(timestamps, desc='OCR frames'):
        frames_gray = []
        half = FRAME_AVERAGE_N // 2
        for offset in range(-half, half + 1):
            ms = (t + offset * dt / FRAME_AVERAGE_N) * 1000
            cap.set(cv2.CAP_PROP_POS_MSEC, ms)
            ret, frame = cap.read()
            if ret:
                frames_gray.append(frame[:, :, 1].astype(np.float32))  # green channel

        if not frames_gray:
            records.append({'time_s': round(t, 3), 'voltage_V': np.nan})
            continue

        # Average frames then convert back to uint8
        avg = np.mean(frames_gray, axis=0).astype(np.uint8)
        proc = preprocess_frame(avg)
        val = ocr_frame(proc)
        records.append({'time_s': round(t, 3), 'voltage_V': val})

    cap.release()
    df = pd.DataFrame(records)

    n_ok  = df['voltage_V'].notna().sum()
    n_bad = df['voltage_V'].isna().sum()
    print(f'OCR: {n_ok} ok, {n_bad} failed ({100*n_bad/len(df):.1f}% NaN) — interpolating NaN values')

    df['voltage_V'] = df['voltage_V'].interpolate(method='linear', limit_direction='both')
    return df


# Run OCR (takes a few minutes for ~878 s of video)
if VOLTAGE_CSV.exists():
    print(f'{VOLTAGE_CSV} already exists — skipping OCR. Delete it to re-run.')
    df_voltage = pd.read_csv(VOLTAGE_CSV)
else:
    df_voltage = extract_voltage_from_video(VIDEO_PATH)
    df_voltage.to_csv(VOLTAGE_CSV, index=False)
    print(f'Saved → {VOLTAGE_CSV}')

df_voltage.head(10)

Video: 878.2 s  |  22.95 fps  |  20153 frames


OCR frames:   0%|          | 8/8783 [00:00<04:11, 34.84it/s]

In [ ]:
# ── Post-OCR cleaning ────────────────────────────────────────────────────────
# Step 1: 18.xxx → 10.xxx  (tens-digit 0→8 confusion)
mask_18 = (df_voltage['voltage_V'] >= 18.0) & (df_voltage['voltage_V'] < 19.0)
df_voltage.loc[mask_18, 'voltage_V'] -= 8.0

# Step 2: subtract 0.8 from known phantom clusters (decimal-digit 0→8 confusion)
ZERO_EIGHT_BANDS = [(11.8, 11.9), (10.8, 10.9)]
for lo, hi in ZERO_EIGHT_BANDS:
    m = (df_voltage['voltage_V'] >= lo) & (df_voltage['voltage_V'] < hi)
    df_voltage.loc[m, 'voltage_V'] -= 0.8

# Step 3: anything still outside the physically valid band → NaN → interpolate
VALID_LO, VALID_HI = 10.85, 11.3
outside = (df_voltage['voltage_V'] < VALID_LO) | (df_voltage['voltage_V'] > VALID_HI)
df_voltage.loc[outside, 'voltage_V'] = np.nan
print(f'Cleaned: {mask_18.sum()} rows 18→10 | {sum((df_voltage["voltage_V"]>=lo)&(df_voltage["voltage_V"]<hi) for lo,hi in ZERO_EIGHT_BANDS)} band-shifted | {outside.sum()} → NaN')
df_voltage['voltage_V'] = df_voltage['voltage_V'].interpolate(method='linear', limit_direction='both')

df_voltage.to_csv(VOLTAGE_CSV, index=False)
print(f'Range: {df_voltage["voltage_V"].min():.3f} – {df_voltage["voltage_V"].max():.3f} V  |  Saved → {VOLTAGE_CSV}')

### Quick sanity check — plot the raw OCR'd voltage signal

### Voltage histogram — identify 0→8 OCR error clusters

The OCR can't reliably distinguish `0` from `8` in 7-segment digits at 64×24 px.
A misread `0→8` adds exactly **+0.8 V** to that digit position.

Look for suspicious mirror clusters offset by 0.8 from a plausible true voltage.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
bins = np.arange(df_voltage['voltage_V'].min() - 0.05,
                 df_voltage['voltage_V'].max() + 0.15, 0.1)
ax.hist(df_voltage['voltage_V'], bins=bins, color='steelblue', edgecolor='white', lw=0.3)
ax.set(xlabel='Voltage (V)', ylabel='Count', title='Raw voltage distribution — look for pairs offset by 0.8 V')
ax.grid(True, lw=0.3, alpha=0.5)
plt.tight_layout()
plt.show()

### Fix 0→8 confusion by subtracting 0.8 from suspicious clusters

For each range where `0→8` produced a phantom cluster, subtract 0.8.
Inspect the histogram above and add entries to `ZERO_EIGHT_BANDS` as needed.

In [ ]:
# ── TUNE THIS ──────────────────────────────────────────────────────────────
# Each entry is a (lo, hi) range where a 0→8 confusion added +0.8 V.
# The code subtracts 0.8 from all readings in that range.
# Confirm these against the histogram above before running.
ZERO_EIGHT_BANDS = [
    (11.8, 11.9),   # true ~11.0 V misread as 11.8 V
    (10.8, 10.9),   # true ~10.0 V misread as 10.8 V  ← remove if 10.8 is a real voltage level
]
# ───────────────────────────────────────────────────────────────────────────

df_voltage = df_voltage.copy()
n_fixed = 0
for lo, hi in ZERO_EIGHT_BANDS:
    band = (df_voltage['voltage_V'] >= lo) & (df_voltage['voltage_V'] < hi)
    df_voltage.loc[band, 'voltage_V'] -= 0.8
    n_fixed += band.sum()
    print(f'  [{lo}, {hi}): {band.sum()} rows corrected → [{lo-0.8:.1f}, {hi-0.8:.1f})')

print(f'\nTotal corrected: {n_fixed}')
df_voltage.to_csv(VOLTAGE_CSV, index=False)
print(f'Saved → {VOLTAGE_CSV}')

# Show updated histogram
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.arange(df_voltage['voltage_V'].min() - 0.05,
                 df_voltage['voltage_V'].max() + 0.15, 0.1)
ax.hist(df_voltage['voltage_V'], bins=bins, color='seagreen', edgecolor='white', lw=0.3)
ax.set(xlabel='Voltage (V)', ylabel='Count', title='Voltage distribution after 0→8 correction')
ax.grid(True, lw=0.3, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Clean OCR voltage signal ─────────────────────────────────────────────────
# Rule 1: 18.xxx → 10.xxx  (OCR misreads '0' as '8' in the tens digit)
# Rule 2: anything whose integer part is not 10 or 11 → NaN → interpolate

df_voltage = df_voltage.copy()

mask_18 = (df_voltage['voltage_V'] >= 18.0) & (df_voltage['voltage_V'] < 19.0)
df_voltage.loc[mask_18, 'voltage_V'] -= 8.0   # 18.xxx → 10.xxx

valid_mask = (
    ((df_voltage['voltage_V'] >= 10.0) & (df_voltage['voltage_V'] < 12.0))
)
n_bad = (~valid_mask).sum()
df_voltage.loc[~valid_mask, 'voltage_V'] = np.nan
print(f'Cleaned: {mask_18.sum()} rows 18→10,  {n_bad} rows → NaN (will interpolate)')

df_voltage['voltage_V'] = df_voltage['voltage_V'].interpolate(method='linear', limit_direction='both')

# Overwrite the OCR CSV with cleaned values
df_voltage.to_csv(VOLTAGE_CSV, index=False)
print(f'Saved cleaned → {VOLTAGE_CSV}')
df_voltage['voltage_V'].describe()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df_voltage['time_s'], df_voltage['voltage_V'], lw=0.8, color='seagreen')
ax.set(xlabel='Video time (s)', ylabel='Voltage (V)', title='OCR voltage — raw (no time offset)')
plt.tight_layout()
plt.show()

## Step 2 — Load current CSV

In [ ]:
df_current = pd.read_csv(CSV_PATH, encoding='utf-16', skiprows=5)
df_current.columns = ['time_s', 'current_uA']
df_current['time_s'] = pd.to_numeric(df_current['time_s'], errors='coerce')
df_current = df_current.dropna().reset_index(drop=True)

print(f'Current CSV: {len(df_current)} rows, t = {df_current["time_s"].min():.2f} – {df_current["time_s"].max():.2f} s')
df_current.head()

## Step 3 — Synchronize with time offset

**`VIDEO_OFFSET_S`**: how many seconds to shift the video timeline relative to the CSV.  
- Positive → video started that many seconds *after* the CSV  
- Negative → video started *before* the CSV  

Adjust until the features line up in the plot below.

In [ ]:
# ── TUNE THIS ──────────────────────────────────────────────────
VIDEO_OFFSET_S = 0.0   # seconds: video_time + offset = csv_time
# ───────────────────────────────────────────────────────────────

# Shift voltage timeline
df_v = df_voltage.copy()
df_v['time_s_aligned'] = df_v['time_s'] + VIDEO_OFFSET_S

# Merge on nearest timestamp (tolerance = half a sample interval)
df_v_sorted = df_v.sort_values('time_s_aligned')
df_c_sorted = df_current.sort_values('time_s')

merged = pd.merge_asof(
    df_c_sorted,
    df_v_sorted[['time_s_aligned', 'voltage_V']].rename(columns={'time_s_aligned': 'time_s'}),
    on='time_s',
    direction='nearest',
    tolerance=0.15   # max 150 ms gap
)

n_matched = merged['voltage_V'].notna().sum()
print(f'Matched rows: {n_matched} / {len(merged)}')
merged.head()

## Step 4 — Plot synchronized data

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

ax1.plot(merged['time_s'], merged['voltage_V'], lw=0.8, color='seagreen', label='Voltage (V)')
ax1.set(ylabel='Voltage (V)', title=f'Synchronized data  |  video offset = {VIDEO_OFFSET_S:+.2f} s')
ax1.legend(loc='upper right')

ax2.plot(merged['time_s'], merged['current_uA'], lw=0.8, color='steelblue', label='Current (µA)')
ax2.set(ylabel='Current (µA)')
ax2.legend(loc='upper right')

# Instantaneous resistance R = V / I  (V, µA → kΩ: multiply by 1e3)
R = merged['voltage_V'] / (merged['current_uA'] * 1e-6)  # Ω
ax3.plot(merged['time_s'], R / 1e3, lw=0.8, color='darkorange', label='Resistance (kΩ)')
ax3.set(xlabel='Time (s)', ylabel='Resistance (kΩ)')
ax3.legend(loc='upper right')

for ax in (ax1, ax2, ax3):
    ax.grid(True, lw=0.3, alpha=0.5)

plt.tight_layout()
plt.show()

## Step 5 — Export merged dataset

In [ ]:
out_path = Path('2026.3.19-actuation2_merged.csv')
merged.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
merged.describe()